![image info](https://raw.githubusercontent.com/albahnsen/MIAD_ML_and_NLP/main/images/banner_1.png)

# Taller: Construcción e implementación de modelos Bagging, Random Forest y XGBoost

En este taller podrán poner en práctica sus conocimientos sobre la construcción e implementación de modelos de Bagging, Random Forest y XGBoost. El taller está constituido por 8 puntos, en los cuales deberan seguir las intrucciones de cada numeral para su desarrollo.

## Datos predicción precio de automóviles

En este taller se usará el conjunto de datos de Car Listings de Kaggle donde cada observación representa el precio de un automóvil teniendo en cuenta distintas variables como año, marca, modelo, entre otras. El objetivo es predecir el precio del automóvil. Para más detalles puede visitar el siguiente enlace: [datos](https://www.kaggle.com/jpayne/852k-used-car-listings).

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Importación de librerías
%matplotlib inline
import pandas as pd

# Lectura de la información de archivo .csv
data = pd.read_csv('https://raw.githubusercontent.com/albahnsen/MIAD_ML_and_NLP/main/datasets/dataTrain_carListings.zip')

# Preprocesamiento de datos para el taller
data = data.loc[data['Model'].str.contains('Camry')].drop(['Make', 'State'], axis=1)
data = data.join(pd.get_dummies(data['Model'], prefix='M'))
data = data.drop(['Model'], axis=1)

# Visualización dataset
data.head()

,Price,Year,Mileage,M_Camry,M_Camry4dr,M_CamryBase,M_CamryL,M_CamryLE,M_CamrySE,M_CamryXLE
7,21995,2014,6480,0,0,0,1,0,0,0
11,13995,2014,39972,0,0,0,0,1,0,0
167,17941,2016,18989,0,0,0,0,0,1,0
225,12493,2014,51330,0,0,0,1,0,0,0
270,7994,2007,116065,0,1,0,0,0,0,0


In [3]:
# Separación de variables predictoras (X) y variable de interés (y)
y = data['Price']
X = data.drop(['Price'], axis=1)

In [4]:
# Separación de datos en set de entrenamiento y test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

### Punto 1 - Árbol de decisión manual

En la celda 1 creen un árbol de decisión **manualmente**  que considere los set de entrenamiento y test definidos anteriormente y presenten el RMSE y MAE del modelo en el set de test.

In [5]:
# Celda 1
import numpy as np
import seaborn as sns
sns.set_style('darkgrid')

# Parámetros del árbol de decisión
max_depth = None # nivel de profundidad del árbol
num_pct = 10 # numero de percentiles para dividir las variables
max_features = None # Usa todas las variables disponibles
min_gain = 0.001 # Solo se hace un split si la ganancia de información es mayor a este valor

In [6]:
# Función para calcular el MSE de un nodo
def mse_nodo(y1):
    if len(y1) == 0:
        return 0
    return ((y1 - y1.mean()) ** 2).mean()

# Función para calcular el RMSE
def rmse_nodo(y1):
    if len(y1) == 0:
        return 0
    return np.sqrt(((y1 - y1.mean()) ** 2).mean())

In [7]:
# Función para encontrar el mejor split de los features basado en la ganancia de información (Menor MSE)

def best_split(X, y, num_pct = 10):

    nro_features = range(X.shape[1]) # Número de features disponibles

    best_split = [0, 0, 0]  # j, split, gain

    for j in nro_features: # j es el índice de la variable a evaluar

        valores_unicos = np.unique(X.iloc[:, j]) # Valores únicos de la variable j

        if len(valores_unicos) <= num_pct: # Si la variable contiene menos valores únicos que el número de percentiles, por eficiencia se evalúan solo los valores únicos de la variable
            splits = valores_unicos[:-1] # Evaluar todos los valores únicos excepto el último
        else:
            splits = np.percentile(X.iloc[:, j], np.arange(0, 100, 100 / (num_pct)).tolist()) # División de la variable en percentiles
            splits = np.unique(splits) # Eliminar valores duplicados
        
        for split in splits:

            filter_l = X.iloc[:,j] < split # Filtro para el nodo izquierdo (Todos los valores de la variable j menores al valor del split)
            y_l = y.loc[filter_l] # Variable respuesta para el nodo izquierdo (Filtro aplicado a la variable respuesta para el nodo izquierdo)
            y_r = y.loc[~filter_l] # Variable respuesta para el nodo derecho (Filtro aplicado a la variable respuesta para el nodo derecho)

            mse_l = mse_nodo(y_l) # MSE del nodo izquierdo
            mse_r = mse_nodo(y_r) # MSE del nodo derecho

            gain = mse_nodo(y) - (len(y_l) * mse_l + len(y_r) * mse_r) / len(y) # Ganancia de información
            if gain > best_split[2]: # Selecciona la mejor ganancia de información en cada iteración
                best_split = [j, split, gain] # Actualiza el mejor split encontrado

    return best_split

j, split, gain = best_split(X, y)
j, split, gain

(0, 2013.0, 8789494.426667094)

In [8]:
def tree_grow(X, y, level = 0, min_gain = 0.001, num_pct = 10, max_depth = None):

    # Si solo es una observación
    if X.shape[0] == 1:
        tree = dict(y_pred = y.iloc[:1].values[0], y_prob = 0.5, level = level, split = -1, n_samples = 1, gain = 0) # split = - 1 define que no hay división, y_prob = 0.5 es la probabilidad de predicción para el nodo hoja
        return tree
    
    # Calcular la mejor división
    j, split, gain = best_split(X, y, num_pct)
    
    # Guardar el árbol y estimar la predicción
    y_pred = y.mean() # Calcula la predicción del nodo como 1 si la media de y es mayor o igual a 0.5, y 0 en caso contrario 
    
    tree = dict(y_pred = y_pred, level = level, split = -1, n_samples = X.shape[0], gain = gain) # Guarda el arbol con la predicción, la probabilidad correspondiente, el nivel del nodo, la división realizada, el número de muestras en el nodo y la ganancia de información obtenida con la división
    # Revisar el criterio de parada 
    if gain < min_gain:
        return tree
    if max_depth is not None:
        if level >= max_depth:
            return tree   
    
    # Continuar creando la partición
    filter_l = X.iloc[:, j] < split
    X_l, y_l = X.loc[filter_l], y.loc[filter_l]
    X_r, y_r = X.loc[~filter_l], y.loc[~filter_l]
    tree['split'] = [j, split]

    # Siguiente iteración para cada partición
    
    tree['sl'] = tree_grow(X_l, y_l, level + 1, min_gain=min_gain, max_depth=max_depth, num_pct=num_pct)
    tree['sr'] = tree_grow(X_r, y_r, level + 1, min_gain=min_gain, max_depth=max_depth, num_pct=num_pct)
    
    return tree

In [9]:
# Aplicación de la función tree_grow
tree = tree_grow(X, y, level=0, min_gain=0.001, max_depth=4, num_pct=10)
tree

{'y_pred': 14538.403716055265,
 'level': 0,
 'split': [0, 2013.0],
 'n_samples': 10495,
 'gain': 8789494.426667094,
 'sl': {'y_pred': 9756.504461221688,
  'level': 1,
  'split': [0, 2011.0],
  'n_samples': 2914,
  'gain': 2852525.302292848,
  'sl': {'y_pred': 8232.290784557908,
   'level': 2,
   'split': [1, 97662.0],
   'n_samples': 1606,
   'gain': 852556.6508446848,
   'sl': {'y_pred': 9363.733644859813,
    'level': 3,
    'split': [0, 2008.0],
    'n_samples': 642,
    'gain': 689117.7883440317,
    'sl': {'y_pred': 8422.825622775801,
     'level': 4,
     'split': -1,
     'n_samples': 281,
     'gain': 379237.7295681741},
    'sr': {'y_pred': 10096.130193905818,
     'level': 4,
     'split': -1,
     'n_samples': 361,
     'gain': 401557.3720880747}},
   'sr': {'y_pred': 7478.778008298756,
    'level': 3,
    'split': [0, 2008.0],
    'n_samples': 964,
    'gain': 313591.5181018696,
    'sl': {'y_pred': 6962.403071017275,
     'level': 4,
     'split': -1,
     'n_samples': 521

In [10]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def tree_predict_single(tree, x):
    # Si es nodo hoja (sin split)
    if tree['split'] == -1:
        return tree['y_pred']
    
    j, split = tree['split']
    
    # Ir a la rama izquierda o derecha
    if x.iloc[j] < split:
        return tree_predict_single(tree['sl'], x)
    else:
        return tree_predict_single(tree['sr'], x)


def tree_predict(tree, X):
    return np.array([tree_predict_single(tree, X.iloc[i]) for i in range(len(X))])

from sklearn.metrics import mean_squared_error, r2_score

pred_manual = tree_predict(tree, X_test)

rmse_manual = np.sqrt(mean_squared_error(y_test, pred_manual))
mae_manual = mean_absolute_error(y_test, pred_manual)
r2_manual = r2_score(y_test, pred_manual)

print(rmse_manual)
print(mae_manual)
print(r2_manual)


1814.145331713227
1357.348894304284
0.7848397020905045


### Punto 2 - Bagging manual

En la celda 2 creen un modelo bagging **manualmente** con 10 árboles de regresión y comenten sobre el desempeño del modelo.

In [11]:
# Celda 2


### Punto 3 - Bagging con librería

En la celda 3, con la librería sklearn, entrenen un modelo bagging con 10 árboles de regresión y el parámetro `max_features` del árbol de decisión igual a `log(n_features)` y comenten sobre el desempeño del modelo.

In [12]:
# Celda 3

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor

n_features = X_train.shape[1]

modelo_DTR = DecisionTreeRegressor(max_features = int(np.log(n_features)))
bagg_DTR = BaggingRegressor(estimator = modelo_DTR, n_estimators = 10, bootstrap= True, random_state = 42)

bagg_DTR.fit(X_train, y_train)


BaggingRegressor(estimator=DecisionTreeRegressor(max_features=2),
                 random_state=42)

In [13]:
pred_bagg_DTR = bagg_DTR.predict(X_test)

rmse_bagg_DTR = np.sqrt(mean_squared_error(y_test, pred_bagg_DTR))
mae_bagg_DTR = mean_absolute_error(y_test, pred_bagg_DTR)
r2_bagg_DTR = r2_score(y_test, pred_bagg_DTR)

print(rmse_bagg_DTR)
print(mae_bagg_DTR)
print(r2_bagg_DTR)

1813.0409476471518
1353.424136561091
0.7851015854948061


### Punto 4 - Random forest con librería

En la celda 4, usando la librería sklearn entrenen un modelo de Randon Forest para regresión  y comenten sobre el desempeño del modelo.

In [14]:
# Celda 4


### Punto 5 - Calibración de parámetros Random forest

En la celda 5, calibren los parámetros max_depth, max_features y n_estimators del modelo de Randon Forest para regresión, comenten sobre el desempeño del modelo y describan cómo cada parámetro afecta el desempeño del modelo.

In [15]:
# Celda 5


### Punto 6 - XGBoost con librería

En la celda 6 implementen un modelo XGBoost de regresión con la librería sklearn y comenten sobre el desempeño del modelo.

In [16]:
# Celda 6


### Punto 7 - Calibración de parámetros XGBoost

En la celda 7 calibren los parámetros learning rate, gamma y colsample_bytree del modelo XGBoost para regresión, comenten sobre el desempeño del modelo y describan cómo cada parámetro afecta el desempeño del modelo.

In [17]:
# Celda 7


### Punto 8 - Comparación y análisis de resultados
En la celda 8 comparen los resultados obtenidos de los diferentes modelos (random forest y XGBoost) y comenten las ventajas del mejor modelo y las desventajas del modelo con el menor desempeño.

In [18]:
# Celda 8
